In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

ratings = pd.read_csv('../data/ratings.csv')
books = pd.read_csv('../data/books.csv')

print("ratings:",ratings.shape)
print("books:",books.shape)
print("Data loaded sucessfully")

ratings: (981756, 3)
books: (10000, 23)
Data loaded sucessfully


In [3]:
# Filter users with fewer than 5 ratings
user_counts = ratings['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index
ratings_filtered = ratings[ratings['user_id'].isin(active_users)]

print(f"Before filtering: {len(ratings):,} ratings, {ratings['user_id'].nunique():,} users")
print(f"After filtering:  {len(ratings_filtered):,} ratings, {ratings_filtered['user_id'].nunique():,} users")
print(f"Users removed: {ratings['user_id'].nunique() - ratings_filtered['user_id'].nunique():,}")

Before filtering: 981,756 ratings, 53,424 users
After filtering:  932,940 ratings, 35,710 users
Users removed: 17,714


In [4]:
# Time-aware train/test split
# Sort by user then we take last 20% of each user's ratings as test
ratings_filtered = ratings_filtered.sort_values(['user_id', 'book_id'])

train_data, test_data = train_test_split(
    ratings_filtered, 
    test_size=0.2, 
    random_state=42,
    stratify=ratings_filtered['user_id'] if False else None
)

print(f"Train set: {len(train_data):,} ratings")
print(f"Test set:  {len(test_data):,} ratings")
print(f"Split ratio: {len(train_data)/len(ratings_filtered):.0%} / {len(test_data)/len(ratings_filtered):.0%}")

Train set: 746,352 ratings
Test set:  186,588 ratings
Split ratio: 80% / 20%


In [5]:
popularity = train_data.groupby('book_id').agg(
    rating_count=('rating','count'),
    rating_mean=('rating','mean')
).reset_index()
popularity = popularity.sort_values('rating_count',ascending=False)

top_10_popular = popularity.head(10).merge(
    books[['id','original_title','authors']],
    left_on='book_id', right_on='id'
)
print("Top 10 most popular books(popularity baseline): ")
print(top_10_popular[['original_title','authors','rating_count','rating_mean']].to_string())

Top 10 most popular books(popularity baseline): 
                                                    original_title                   authors  rating_count  rating_mean
0                                                      Term Limits               Vince Flynn            94     4.255319
1                                                        After You                Jojo Moyes            93     3.408602
2                                                      Fool's Fate                Robin Hobb            92     4.217391
3  Raise High the Roof Beam, Carpenters / Seymour: An Introduction             J.D. Salinger            92     4.141304
4                                              Ut og stjæle hester  Per Petterson, Anne Born            91     3.714286
5                         Micah (Anita Blake, Vampire Hunter, #13)       Laurell K. Hamilton            91     3.439560
6                                        The Gifts of Imperfection               Brené Brown            91     

In [11]:
def precision_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k=10):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

popular_books = popularity['book_id'].tolist()

precisions, recalls = [], []

test_users = test_data['user_id'].unique()[:500]

for user_id in test_users:
    relevant = test_data[test_data['user_id'] == user_id]['book_id'].tolist()
    precision = precision_at_k(popular_books, relevant)
    recall = recall_at_k(popular_books, relevant)
    precisions.append(precision)
    recalls.append(recall)

print(f"Popularity Baseline Results:")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Recall@10:    {np.mean(recalls):.4f}")

Popularity Baseline Results:
Precision@10: 0.0004
Recall@10:    0.0001


In [12]:
from surprise import Dataset, Reader, KNNBasic
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(
    train_data[['user_id', 'book_id', 'rating']], 
    reader
)

print("Running item-item CF with cross validation...")
algo = KNNBasic(sim_options={'user_based': False})
results = cross_validate(algo, surprise_data, measures=['RMSE', 'MAE'], cv=3, verbose=True)

print(f"\nMean RMSE: {results['test_rmse'].mean():.4f}")
print(f"Mean MAE:  {results['test_mae'].mean():.4f}")

Running item-item CF with cross validation...
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    0.9050  0.9087  0.9069  0.9069  0.0015  
MAE (testset)     0.6750  0.6787  0.6764  0.6767  0.0015  
Fit time          2.70    2.31    2.38    2.46    0.17    
Test time         6.74    6.30    6.05    6.36    0.29    

Mean RMSE: 0.9069
Mean MAE:  0.6767


# Baseline Models Summary

## Models Built
- Popularity baseline — recommend globally most rated books
- Item-item CF — KNNBasic via Surprise library, 3-fold CV

## Results
| Model | Metric | Score |
|---|---|---|
| Popularity | Precision@10 | 0.0004 |
| Popularity | Recall@10 | 0.0001 |
| Item-Item CF | RMSE | 0.9069 |
| Item-Item CF | MAE | 0.6767 |

## Key Learnings
- Popularity baseline is near zero — low bar to beat
- Item-item CF achieves 0.9 RMSE — reasonable but not impressive
- RMSE measures rating prediction, not ranking quality
- Next models evaluated on precision@10 and recall@10

## Next Step
- 03_matrix_factorisation.ipynb — nn.Embedding + dot product in PyTorch